In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')

df = load_dataset('크루즈입항정보')

In [ ]:
# =========================================================
# 1. 데이터 전처리
# =========================================================

berth_source = df.copy()
berth_source.columns = berth_source.columns.str.strip()

berth_col = "계선장소.2"

required_cols = [
    berth_col,
    "입출",
    "선박용도",
    "입항일시",
    "승객"
]

missing_cols = [
    col for col in required_cols
    if col not in berth_source.columns
]

if missing_cols:
    raise ValueError(
        f"다음 컬럼을 확인해주세요: {missing_cols}"
    )


# 입항한 크루즈선만 필터링
berth_data = berth_source[
    (berth_source["입출"] == "입항") &
    (berth_source["선박용도"] == "크루즈선")
].copy()


# 입항일시 변환
berth_data["입항일시"] = pd.to_datetime(
    berth_data["입항일시"],
    errors="coerce"
)


# 승객수 숫자형 변환
berth_data["승객"] = pd.to_numeric(
    berth_data["승객"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)


# 분석 기간: 2013-01-01 ~ 2026-10-16
berth_data = berth_data[
    (berth_data["입항일시"] >= "2013-01-01") &
    (berth_data["입항일시"] < "2026-10-17")
].copy()


if berth_data.empty:
    raise ValueError(
        "분석 기간에 해당하는 데이터가 없습니다."
    )


# =========================================================
# 2. 계선장소 정리 및 TOP 5 선정
# =========================================================

berth_data["계선장소표시"] = (
    berth_data[berth_col]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna("계선장소 미상")
)


# 계선장소별 입항건수 집계
place_counts = (
    berth_data
    .groupby("계선장소표시")
    .size()
    .reset_index(name="입항건수")
    .sort_values(
        ["입항건수", "계선장소표시"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


# 입항건수 기준 TOP 5
top5_names = (
    place_counts
    .head(5)["계선장소표시"]
    .tolist()
)


# TOP 5 이외의 장소는 기타로 통합
berth_data["표시그룹"] = (
    berth_data["계선장소표시"]
    .where(
        berth_data["계선장소표시"].isin(top5_names),
        "기타"
    )
)


has_other = (
    ~berth_data["계선장소표시"].isin(top5_names)
).any()


place_order = top5_names + (
    ["기타"] if has_other else []
)


# =========================================================
# 3. 계선장소별 입항건수 및 승객수 집계
# =========================================================

chart_data = (
    berth_data
    .groupby("표시그룹")
    .agg(
        입항건수=("입항일시", "size"),
        승객수합계=(
            "승객",
            lambda x: x.sum(min_count=1)
        ),
        승객결측건수=(
            "승객",
            lambda x: x.isna().sum()
        )
    )
    .reindex(place_order)
    .reset_index()
)


# 승객수 단위를 만 명으로 변환
chart_data["승객수_만명"] = (
    chart_data["승객수합계"] / 10_000
)


# 장소명에 괄호가 있으면 괄호 앞에서 줄바꿈
chart_data["그래프장소명"] = (
    chart_data["표시그룹"]
    .str.replace("(", "<br>(", regex=False)
)


# TOP 5와 기타 구분
is_top5 = chart_data["표시그룹"] != "기타"


# 개별 장소 중 지표별 최댓값
max_arrivals = chart_data.loc[
    is_top5, "입항건수"
].max()

max_passengers = chart_data.loc[
    is_top5, "승객수합계"
].max()


# =========================================================
# 4. 막대 색상 설정
# =========================================================

# 최댓값: 주황색
# 나머지 TOP 5: 파란색
# 기타: 회색

arrival_colors = [
    "#A6AFBA" if place == "기타"
    else "#E67E22" if count == max_arrivals
    else "#3478C0"
    for place, count in zip(
        chart_data["표시그룹"],
        chart_data["입항건수"]
    )
]


passenger_colors = [
    "#A6AFBA" if place == "기타"
    else "#E67E22"
    if pd.notna(value)
    and pd.notna(max_passengers)
    and value == max_passengers
    else "#3478C0"
    for place, value in zip(
        chart_data["표시그룹"],
        chart_data["승객수합계"]
    )
]


# =========================================================
# 5. 데이터 라벨 및 비중 계산
# =========================================================

total_arrivals = chart_data["입항건수"].sum()

total_passengers = chart_data["승객수합계"].sum(
    min_count=1
)


if total_arrivals <= 0:
    raise ValueError("전체 입항건수가 0입니다.")


# 입항건수 비중
chart_data["입항비중"] = (
    chart_data["입항건수"] / total_arrivals * 100
)


# 승객수 비중
if pd.notna(total_passengers) and total_passengers > 0:
    chart_data["승객비중"] = (
        chart_data["승객수합계"] / total_passengers * 100
    )
else:
    chart_data["승객비중"] = float("nan")


# 입항건수: 정수 + 비중
arrival_labels = [
    f"{count:,.0f}건 ({share:.1f}%)"
    for count, share in zip(
        chart_data["입항건수"],
        chart_data["입항비중"]
    )
]


# 승객수: 만 명 또는 실제 인원 + 비중
def make_passenger_label(value, share):

    if pd.isna(value):
        return ""

    if value >= 10_000:
        value_text = f"{value / 10_000:,.1f}만 명"
    else:
        value_text = f"{value:,.0f}명"

    if pd.notna(share):
        return f"{value_text} ({share:.1f}%)"

    return value_text


passenger_labels = [
    make_passenger_label(value, share)
    for value, share in zip(
        chart_data["승객수합계"],
        chart_data["승객비중"]
    )
]


# =========================================================
# 6. 공통 그래프 스타일
# =========================================================

footer_text = (
    "※ 분석 기간: 2013.01.01–2026.10.16"
    "<br>※ 입항건수 기준 TOP 5 및 기타이며, 기타는 나머지 장소의 합계입니다."
    "<br>※ 막대 옆 괄호는 전체 입항건수 또는 전체 승객수 중 해당 그룹의 비중입니다."
    "<br>※ 주황색은 개별 장소 중 해당 지표의 최댓값, 회색은 기타를 의미합니다."
)


# 추가 주석을 받아 하단에 한 번만 표시하는 함수
def apply_berth_layout(fig, title_text, extra_footer=""):

    combined_footer = footer_text

    if extra_footer:
        combined_footer += "<br>" + extra_footer

    fig.update_layout(
        title=dict(
            text=(
                f"<b>{title_text}</b>"
                "<br>"
                "<span style='font-size:16px;color:#657487'>"
                "2013.01.01–2026.10.16 · 입항건수 기준 TOP 5 및 기타"
                "</span>"
            ),
            x=0.5,
            y=0.94,
            xanchor="center",
            yanchor="top",
            font=dict(size=23)
        ),
        template="plotly_white",
        width=1200,
        height=675,
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(
            family="Malgun Gothic, Arial, sans-serif",
            size=16,
            color="#243247"
        ),
        margin=dict(
            l=260,
            r=100,
            t=110,
            b=175
        ),
        bargap=0.35,
        showlegend=False
    )

    fig.update_yaxes(
        title=None,
        categoryorder="array",
        categoryarray=chart_data["그래프장소명"].tolist(),
        autorange="reversed",
        showgrid=False,
        automargin=True,
        tickfont=dict(size=15)
    )

    # 하단 주석은 그래프당 하나만 추가
    fig.add_annotation(
        x=0,
        y=-0.22,
        xref="paper",
        yref="paper",
        text=combined_footer,
        showarrow=False,
        xanchor="left",
        yanchor="top",
        align="left",
        font=dict(
            size=12,
            color="#657487"
        )
    )


# =========================================================
# 그래프 1. 계선장소별 입항건수
# =========================================================

fig_berth_arrivals = go.Figure()


fig_berth_arrivals.add_trace(
    go.Bar(
        x=chart_data["입항건수"],
        y=chart_data["그래프장소명"],
        orientation="h",
        marker_color=arrival_colors,
        marker_line_width=0,
        text=arrival_labels,
        textposition="outside",
        textfont=dict(
            size=17,
            color="#243247"
        ),
        cliponaxis=False,
        customdata=chart_data[
            ["표시그룹", "입항비중"]
        ].to_numpy(),
        hovertemplate=(
            "<b>%{customdata[0]}</b>"
            "<br>입항건수: %{x:,.0f}건"
            "<br>전체 입항 중 비중: %{customdata[1]:.1f}%"
            "<extra></extra>"
        )
    )
)


fig_berth_arrivals.update_xaxes(
    title=dict(
        text="입항건수 (건)",
        font=dict(size=16),
        standoff=12
    ),
    range=[
        0,
        chart_data["입항건수"].max() * 1.55
    ],
    tickformat=",d",
    gridcolor="#E9EDF2",
    zeroline=False
)


apply_berth_layout(
    fig_berth_arrivals,
    "부산항 크루즈 계선장소별 입항건수"
)


fig_berth_arrivals.show(
    config={
        "displaylogo": False,
        "toImageButtonOptions": {
            "format": "png",
            "filename": "PPT_계선장소별_입항건수_TOP5_기타",
            "width": 1200,
            "height": 675,
            "scale": 3
        }
    }
)


# =========================================================
# 그래프 2. 계선장소별 승객수 합계
# =========================================================

fig_berth_passengers = go.Figure()


fig_berth_passengers.add_trace(
    go.Bar(
        x=chart_data["승객수_만명"],
        y=chart_data["그래프장소명"],
        orientation="h",
        marker_color=passenger_colors,
        marker_line_width=0,
        text=passenger_labels,
        textposition="outside",
        textfont=dict(
            size=17,
            color="#243247"
        ),
        cliponaxis=False,
        customdata=chart_data[
            [
                "표시그룹",
                "승객수합계",
                "승객결측건수",
                "승객비중"
            ]
        ].to_numpy(),
        hovertemplate=(
            "<b>%{customdata[0]}</b>"
            "<br>승객수 합계: %{customdata[1]:,.0f}명"
            "<br>전체 승객수 중 비중: %{customdata[3]:.1f}%"
            "<br>승객 결측 기록: %{customdata[2]:,.0f}건"
            "<extra></extra>"
        )
    )
)


passenger_max = chart_data["승객수_만명"].max()

passenger_limit = (
    max(1, passenger_max * 1.55)
    if pd.notna(passenger_max)
    else 1
)


fig_berth_passengers.update_xaxes(
    title=dict(
        text="승객수 합계 (만 명)",
        font=dict(size=16),
        standoff=12
    ),
    range=[0, passenger_limit],
    tickformat=",.1f",
    gridcolor="#E9EDF2",
    zeroline=False
)


# 하단 결측값 안내도 공통 주석에 통합
apply_berth_layout(
    fig_berth_passengers,
    "부산항 크루즈 계선장소별 승객수 합계",
    extra_footer=(
        "※ 승객수 합계는 승객 결측값을 제외하여 계산했습니다."
    )
)


fig_berth_passengers.show(
    config={
        "displaylogo": False,
        "toImageButtonOptions": {
            "format": "png",
            "filename": "PPT_계선장소별_승객수합계_TOP5_기타",
            "width": 1200,
            "height": 675,
            "scale": 3
        }
    }
)